# Model selection via cross-validation

## 1. Setup

In [ ]:
# packages

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.calibration import calibration_curve

In [ ]:
# Run from the project root or from the 02 Scripts folder.
folder_path = Path.cwd()
if folder_path.name == "02 Scripts":
    folder_path = folder_path.parent

input_path = folder_path / "01 Input"
temp_path = folder_path / "03 Temp"
table_path = folder_path / "04 Tables"
chart_path = folder_path / "05 Charts"

if not input_path.is_dir():
    raise FileNotFoundError(f"Input folder not found: {input_path}")
for output_path in (temp_path, table_path, chart_path):
    output_path.mkdir(exist_ok=True)

In [ ]:
# load data

training_df = pd.read_csv(temp_path / "training_cleaned.csv")
test_df = pd.read_csv(temp_path / "test_cleaned.csv")

## 2. Initial definitions

In [ ]:
#Split training data into features and target

X_train = training_df.drop("poor", axis=1)
y_train = training_df["poor"]

X_test = test_df.drop("poor", axis=1)
y_test = test_df["poor"]

In [ ]:
# define two sets of features

categorical_cols = ["kjkrfgld", "bpowgknt", "raksnhjf", "vwpsxrgk"]
numeric_cols = ["omtioxzz", "yfmzwkru", "tiwrsloh"]

feature_cols = numeric_cols + categorical_cols

assert set(feature_cols) == set(X_train.columns)
assert set(feature_cols) == set(X_test.columns)

## 3 Preprocessing 

In [ ]:
# as imputing and rescaling use parameters learnt from the data (e.g., median, mean, sd)
# i put them as part of the pipeline so that these parameters are re-learnt from the training fold only
# rather than partially learnt from the validation fold too. this prevents leakage

# for numeric variables, impute with the median and add an indicator when imputed
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ]
)

# preprocessor for unregularised logit
# drop first category to avoid perfect collinearity with the intercept
# however, this means unknown categories in the validation fold (if they appear)
# will be treated as though they are the same as the baseline

categorical_transformer_drop_first = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]
)

# preprocessor for regularised logit and tree models
# now unseen categories are encoded as all zeroes for that variable's one-hot columns
# i.e. no category-specific effect is added for that variable

categorical_transformer_full = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# logit preprocessor
preprocessor_drop_first = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer_drop_first, categorical_cols)
    ]
)

# other models preprocessor
preprocessor_full = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer_full, categorical_cols)
    ]
)

In [ ]:
# 5-fold stratfied CV (so target is balanced in each fold). 
# #shuffle in case non-random ordering in the dataset; fix random state so shuffling consistent each time


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1
)

## 5. Models

In [ ]:
# unregularised logit
# use drop-first preprocessor to avoid perfect collinearity between encoded vars

logit_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_drop_first),
        ("model", LogisticRegression(
            C=np.inf,
            solver="lbfgs",
            max_iter=5000
        ))
    ]
)


# elastic net logit
# use preprocessor for non-logit models
elastic_net_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_full),
        ("model", LogisticRegression(
            l1_ratio=0.5,  # overridden during cross validation
            solver="saga",
            max_iter=10000,
            random_state=1
        ))
    ]
)


# random forest
# use preprocessor for non-logit models
rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_full),
        ("model", RandomForestClassifier(
            n_estimators = 500,
            random_state=1,
            n_jobs=-1
        ))
    ]
)


# xgboost

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_full),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=1,
            n_jobs=-1
        ))
    ]
)

## 6. Cross-validated model comparison

In [ ]:
# model tuning and comparison
# evaluate grid candidates sequentially; RF and XGBoost parallelise individual fits
# models are trained using the 5 fold CV set and evaluated using average ROC AUC across the 5 validation folds
# tuning grids are deliberately small so that the code is runnable locally

rows = []

# unregularised logit
# obviously, no hyperparamters to tune
# use area under the ROC to score the models
logit_scores = cross_val_score(
    logit_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

rows.append({
    "model": "Unregularised logit",
    "mean_cv_auc": logit_scores.mean(),
    "sd_cv_auc": logit_scores.std(),
    "best_params": None
})


# elastic net logit
# tune both the overall penalty term and the relative weight for lasso and ridge
elastic_net_grid = {
    "model__C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "model__l1_ratio": [0.1, 0.5, 0.9]
}

elastic_net_search = GridSearchCV(
    estimator=elastic_net_pipeline,
    param_grid=elastic_net_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

elastic_net_search.fit(X_train, y_train)

rows.append({
    "model": "Elastic net logit",
    "mean_cv_auc": elastic_net_search.best_score_,
    "sd_cv_auc": elastic_net_search.cv_results_["std_test_score"][elastic_net_search.best_index_],
    "best_params": elastic_net_search.best_params_
})


# random forest
# tune 4 different parameters
# number of trees, tree depth, minimum observations per leaf, number of features randomly selected as candidates per split
rf_grid = {
    "model__max_depth": [None, 3, 5, 10],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

rf_search.fit(X_train, y_train)

rows.append({
    "model": "Random forest",
    "mean_cv_auc": rf_search.best_score_,
    "sd_cv_auc": rf_search.cv_results_["std_test_score"][rf_search.best_index_],
    "best_params": rf_search.best_params_
})


# xgboost
# tune main parameters including number of trees, tree depth, learning rate, and row/column sampling
xgb_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [2, 3, 5],
    "model__learning_rate": [0.03, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

xgb_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=xgb_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

xgb_search.fit(X_train, y_train)

rows.append({
    "model": "XGBoost",
    "mean_cv_auc": xgb_search.best_score_,
    "sd_cv_auc": xgb_search.cv_results_["std_test_score"][xgb_search.best_index_],
    "best_params": xgb_search.best_params_
})

# append to results dataframe, sort descending by performance
cv_results = pd.DataFrame(rows)
cv_results = cv_results.sort_values("mean_cv_auc", ascending=False)

display(cv_results)

# export CV comparison table

cv_results.to_excel(table_path / "cv_model_comparison.xlsx", index=False)


## 7. Final model evaluation on the test set only

In [ ]:
# logit performs only marginally worse than the other 3 models
# given business context (prediction for the world bank)
# choose logit as default given similar performance and easier interpretation for WB economists
# however, still also evaluate the best performing model (XGB) on the test data too for completeness 

# final models
final_logit = logit_pipeline
final_xgb = xgb_search.best_estimator_

# fit on entire training data
final_logit.fit(X_train, y_train)
final_xgb.fit(X_train, y_train)

# obtain predicted probabilities
logit_test_prob = final_logit.predict_proba(X_test)[:, 1]
xgb_test_prob = final_xgb.predict_proba(X_test)[:, 1]

# table of ROC AUC scores from the test data
test_results = pd.DataFrame({
    "model": ["Unregularised logit", "XGBoost"],
    "test_auc": [
        roc_auc_score(y_test, logit_test_prob),
        roc_auc_score(y_test, xgb_test_prob)
    ]
})

display(test_results)

test_results.to_excel(table_path / "test_model_comparison.xlsx", index=False)

In [ ]:
# plot ROC curve (i.e. TPR vs FPR for different thresholds) for either model
# using the predicted probabilities
logit_fpr, logit_tpr, _ = roc_curve(y_test, logit_test_prob)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_test_prob)

plt.figure()
plt.plot(logit_fpr, logit_tpr, label="Logit")
plt.plot(xgb_fpr, xgb_tpr, label="XGBoost")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend()
plt.tight_layout()

plt.savefig(chart_path / "test_roc_logit_xgboost.png")
plt.show()

In [ ]:
# estimate threshold which maximises Youden's J for the logit model using CV
# CV obtains predicted probabilities for observations when they are in the held out validation set
# this means that estimated probabilities come from a model which has not used those observations for training
# this prevents probabilities being over optimistic, which is important for selecting thresholds
# this more accurately reflects how the model would predict probabilities on unseen data

logit_oof_prob = cross_val_predict(
    logit_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

# calculate YJ, and threshold which maximises it
fpr, tpr, thresholds = roc_curve(y_train, logit_oof_prob)

youden_j = tpr - fpr
youden_threshold = thresholds[youden_j.argmax()]

print("Youden threshold:", youden_threshold)

In [ ]:
# output confusion matrices using the test data for different thresholds (including youden j maximiser)

# thresholds to loop over
thresholds_to_check = [0.25, 0.5, 0.75, youden_threshold]

with pd.ExcelWriter(table_path / "logit_confusion_matrices.xlsx") as writer:

    for c in thresholds_to_check:

        y_pred = logit_test_prob >= c

        # confusion matrix
        cm = confusion_matrix(y_test, y_pred)

        cm_df = pd.DataFrame(
            cm,
            index=["Actual non-poor", "Actual poor"],
            columns=["Predicted non-poor", "Predicted poor"]
        )

        display(cm_df)

        sheet_name = f"th_{c:.3f}"
        cm_df.to_excel(writer, sheet_name=sheet_name)


In [ ]:
# calibration plots on the test data
# ==> bin predicted probabilities (x axis) and compare with actual poverty rates in the test data
# compare logit and XGB

logit_prob_true, logit_prob_pred = calibration_curve(
    y_test,
    logit_test_prob,
    n_bins=10,
    strategy="quantile"
)

xgb_prob_true, xgb_prob_pred = calibration_curve(
    y_test,
    xgb_test_prob,
    n_bins=10,
    strategy="quantile"
)

plt.figure()
plt.plot(logit_prob_pred, logit_prob_true, marker="o", label="Logit")
plt.plot(xgb_prob_pred, xgb_prob_true, marker="o", label="XGBoost")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed poverty rate")
plt.legend()
plt.tight_layout()

plt.savefig(chart_path / "calibration_logit_xgboost.png")
plt.show()


In [ ]:
# Brier scores (essentially MSE)

calibration_results = pd.DataFrame({
    "model": ["Unregularised logit", "XGBoost"],
    "brier_score": [
        brier_score_loss(y_test, logit_test_prob),
        brier_score_loss(y_test, xgb_test_prob)
    ]
})

display(calibration_results)

calibration_results.to_excel(table_path / "brier_scores.xlsx", index=False)
